# Novel Uncertainty-Aware Stacking (7 Models) - After Tuning

This notebook evaluates seven after-tuning Novel Uncertainty-Aware Stacking configurations using 5-fold stratified cross-validation.

The base learners use the tuned hyperparameters obtained from Bayesian optimization, and the Logistic Regression meta-learner is also configured with tuned hyperparameters.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from imblearn.combine import SMOTEENN

CV_RANDOM_STATE = 42
MODEL_RANDOM_STATE = 0
TARGET_COLUMN = "HeartDisease"


def entropy_binary(p, eps=1e-12):
    """Compute binary entropy from predicted probabilities."""
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def rf_vote_variance(rf_pipeline, X):
    """Compute Random Forest vote variance as an uncertainty feature."""
    scaler = rf_pipeline.named_steps["standardscaler"]
    rf_model = rf_pipeline.named_steps["randomforestclassifier"]

    X_scaled = scaler.transform(X)
    votes = np.column_stack([tree.predict(X_scaled) for tree in rf_model.estimators_])
    p_hat = votes.mean(axis=1)

    return p_hat * (1.0 - p_hat)


def build_base_model(model_name, params):
    """Create a base learner pipeline."""
    if model_name == "LR":
        estimator = LogisticRegression(**params)
    elif model_name == "SVM":
        estimator = SVC(probability=True, **params)
    elif model_name == "RF":
        estimator = RandomForestClassifier(**params)
    elif model_name == "ET":
        estimator = ExtraTreesClassifier(**params)
    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    return make_pipeline(StandardScaler(), estimator)


def get_meta_features(fitted_models, X):
    """
    Generate uncertainty-aware meta-features.

    LR, SVM, and ET:
        predicted probability + entropy
    RF:
        predicted probability + vote variance
    """
    features = []

    for model_name, model in fitted_models.items():
        prob = model.predict_proba(X)[:, 1]

        if model_name == "RF":
            uncertainty = rf_vote_variance(model, X)
        else:
            uncertainty = entropy_binary(prob)

        features.extend([prob, uncertainty])

    return np.column_stack(features)


def evaluate_predictions(y_true, y_pred, y_proba):
    """Compute classification performance metrics."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1_score": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_proba),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }


def run_novel_stacking_cv(X, y, config, n_splits=5):
    """Run 5-fold cross-validation for one novel stacking configuration."""
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=CV_RANDOM_STATE,
    )

    fold_results = []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Apply SMOTEENN only to the training fold
        smoteenn = SMOTEENN(random_state=MODEL_RANDOM_STATE)
        X_train_resampled, y_train_resampled = smoteenn.fit_resample(X_train, y_train)

        # Train base learners
        fitted_models = {}
        for model_name in config["base_models"]:
            fitted_models[model_name] = build_base_model(
                model_name,
                config["base_params"][model_name],
            )
            fitted_models[model_name].fit(X_train_resampled, y_train_resampled)

        # Create meta-features
        X_meta_train = get_meta_features(fitted_models, X_train_resampled)
        X_meta_test = get_meta_features(fitted_models, X_test)

        # Train meta-learner
        meta_model = LogisticRegression(**config["meta_params"])
        meta_model.fit(X_meta_train, y_train_resampled)

        # Predict on the original test fold
        y_proba = meta_model.predict_proba(X_meta_test)[:, 1]
        y_pred = meta_model.predict(X_meta_test)

        row = evaluate_predictions(y_test, y_pred, y_proba)
        row["Fold"] = fold_idx
        row["Model"] = config["name"]
        fold_results.append(row)

    return pd.DataFrame(fold_results)


def summarize_results(all_fold_results):
    """Summarize fold-level results by model."""
    metric_cols = ["Accuracy", "Precision", "Recall", "F1_score", "AUC"]

    summary = (
        all_fold_results
        .groupby("Model")[metric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )

    return summary


In [ ]:
# Load dataset
DATA_PATH = "../Data/Partition_class0_cluster5_cleaned_encoded_standardized.csv"

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET_COLUMN]).to_numpy()
y = df[TARGET_COLUMN].to_numpy().astype(int)

In [ ]:
MODEL_CONFIGS = [
    {
        "name": "LR+ET",
        "base_models": ["LR", "ET"],
        "base_params": {
            "LR": {
                "penalty": "l1",
                "solver": "saga",
                "C": 0.010289782695947526,
                "max_iter": 20,
                "random_state": MODEL_RANDOM_STATE,
            },
            "ET": {
                "n_estimators": 104,
                "max_depth": 28,
                "min_samples_split": 10,
                "min_samples_leaf": 10,
                "max_features": "sqrt",
                "bootstrap": False,
                "random_state": MODEL_RANDOM_STATE,
            },
        },
        "meta_params": {
            "C": 0.00010592264248535084,
            "max_iter": 1905,
            "random_state": MODEL_RANDOM_STATE,
        },
    },
    {
        "name": "RF+ET",
        "base_models": ["RF", "ET"],
        "base_params": {
            "RF": {
                "n_estimators": 268,
                "max_depth": 5,
                "min_samples_split": 14,
                "min_samples_leaf": 5,
                "max_features": "sqrt",
                "bootstrap": True,
                "random_state": MODEL_RANDOM_STATE,
            },
            "ET": {
                "n_estimators": 188,
                "max_depth": 40,
                "min_samples_split": 13,
                "min_samples_leaf": 3,
                "max_features": "sqrt",
                "bootstrap": True,
                "random_state": MODEL_RANDOM_STATE,
            },
        },
        "meta_params": {
            "C": 0.0001249238353973739,
            "max_iter": 716,
            "random_state": MODEL_RANDOM_STATE,
        },
    },
    {
        "name": "SVM+ET",
        "base_models": ["SVM", "ET"],
        "base_params": {
            "SVM": {
                "C": 0.013319421470729472,
                "kernel": "linear",
                "gamma": "scale",
                "random_state": MODEL_RANDOM_STATE,
            },
            "ET": {
                "n_estimators": 122,
                "max_depth": 3,
                "min_samples_split": 15,
                "min_samples_leaf": 9,
                "max_features": "log2",
                "bootstrap": False,
                "random_state": MODEL_RANDOM_STATE,
            },
        },
        "meta_params": {
            "C": 0.00010301783397406505,
            "max_iter": 959,
            "random_state": MODEL_RANDOM_STATE,
        },
    },
    {
        "name": "RF+LR+ET",
        "base_models": ["RF", "LR", "ET"],
        "base_params": {
            "RF": {
                "n_estimators": 194,
                "max_depth": 5,
                "min_samples_split": 10,
                "min_samples_leaf": 10,
                "max_features": "log2",
                "bootstrap": True,
                "random_state": MODEL_RANDOM_STATE,
            },
            "LR": {
                "C": 0.42211347466728355,
                "max_iter": 1501,
                "random_state": MODEL_RANDOM_STATE,
            },
            "ET": {
                "n_estimators": 109,
                "max_depth": 50,
                "min_samples_split": 17,
                "min_samples_leaf": 5,
                "max_features": "sqrt",
                "bootstrap": False,
                "random_state": MODEL_RANDOM_STATE,
            },
        },
        "meta_params": {
            "C": 0.00015234915477268398,
            "max_iter": 242,
            "random_state": MODEL_RANDOM_STATE,
        },
    },
    {
        "name": "RF+SVM+ET",
        "base_models": ["RF", "SVM", "ET"],
        "base_params": {
            "RF": {
                "n_estimators": 196,
                "max_depth": 37,
                "min_samples_split": 4,
                "min_samples_leaf": 6,
                "max_features": "log2",
                "bootstrap": True,
                "random_state": MODEL_RANDOM_STATE,
            },
            "SVM": {
                "C": 0.08481460852376649,
                "kernel": "rbf",
                "gamma": "auto",
                "random_state": MODEL_RANDOM_STATE,
            },
            "ET": {
                "n_estimators": 185,
                "max_depth": 3,
                "min_samples_split": 13,
                "min_samples_leaf": 4,
                "max_features": "sqrt",
                "bootstrap": True,
                "random_state": MODEL_RANDOM_STATE,
            },
        },
        "meta_params": {
            "C": 0.00010301783397406505,
            "max_iter": 959,
            "random_state": MODEL_RANDOM_STATE,
        },
    },
    {
        "name": "LR+SVM+ET",
        "base_models": ["LR", "SVM", "ET"],
        "base_params": {
            "LR": {
                "C": 1.6983241510357745,
                "max_iter": 1615,
                "random_state": MODEL_RANDOM_STATE,
            },
            "SVM": {
                "C": 0.08481460852376649,
                "kernel": "rbf",
                "gamma": "auto",
                "random_state": MODEL_RANDOM_STATE,
            },
            "ET": {
                "n_estimators": 55,
                "max_depth": 40,
                "min_samples_split": 3,
                "min_samples_leaf": 10,
                "max_features": "log2",
                "bootstrap": False,
                "random_state": MODEL_RANDOM_STATE,
            },
        },
        "meta_params": {
            "C": 0.00010301783397406505,
            "max_iter": 959,
            "random_state": MODEL_RANDOM_STATE,
        },
    },
    {
        "name": "LR+SVM+RF+ET",
        "base_models": ["LR", "SVM", "RF", "ET"],
        "base_params": {
            "LR": {
                "C": 0.6141568368877525,
                "max_iter": 451,
                "random_state": MODEL_RANDOM_STATE,
            },
            "SVM": {
                "C": 0.30695155161630694,
                "kernel": "linear",
                "gamma": "auto",
                "random_state": MODEL_RANDOM_STATE,
            },
            "RF": {
                "n_estimators": 59,
                "max_depth": 21,
                "min_samples_split": 6,
                "min_samples_leaf": 8,
                "max_features": "log2",
                "bootstrap": True,
                "random_state": MODEL_RANDOM_STATE,
            },
            "ET": {
                "n_estimators": 81,
                "max_depth": 30,
                "min_samples_split": 5,
                "min_samples_leaf": 9,
                "max_features": "log2",
                "bootstrap": True,
                "random_state": MODEL_RANDOM_STATE,
            },
        },
        "meta_params": {
            "C": 0.00010294351475169947,
            "max_iter": 692,
            "random_state": MODEL_RANDOM_STATE,
        },
    },
]

print(f"Number of model configurations: {len(MODEL_CONFIGS)}")
